In [ ]:
import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt

# ============================================================
# Bayesian Model Averaging: Linear vs Quadratic Regression
# ============================================================

# Example data
np.random.seed(42)
x = np.linspace(-3, 3, 50)
y = 2 + 1.5*x + 0.8*x**2 + np.random.normal(0, 1, len(x))

# Scale x for stable sampling
x_scaled = (x - x.mean()) / x.std()

# ------------------------------------------------------------
# 1. Linear model
# ------------------------------------------------------------
with pm.Model() as linear_model:

    alpha = pm.Normal("alpha", mu=0, sigma=10)
    beta = pm.Normal("beta", mu=0, sigma=10)
    sigma = pm.HalfNormal("sigma", sigma=5)

    mu = alpha + beta * x_scaled

    y_obs = pm.Normal(
        "y_obs",
        mu=mu,
        sigma=sigma,
        observed=y
    )

    trace_linear = pm.sample(
        1000,
        tune=1000,
        return_inferencedata=True,
        random_seed=42,
        target_accept=0.9
    )

# ------------------------------------------------------------
# 2. Quadratic model
# ------------------------------------------------------------
with pm.Model() as quad_model:

    alpha = pm.Normal("alpha", mu=0, sigma=10)
    beta = pm.Normal("beta", mu=0, sigma=10)
    beta2 = pm.Normal("beta2", mu=0, sigma=10)
    sigma = pm.HalfNormal("sigma", sigma=5)

    mu = (
        alpha
        + beta * x_scaled
        + beta2 * x_scaled**2
    )

    y_obs = pm.Normal(
        "y_obs",
        mu=mu,
        sigma=sigma,
        observed=y
    )

    trace_quad = pm.sample(
        1000,
        tune=1000,
        return_inferencedata=True,
        random_seed=42,
        target_accept=0.9
    )

# ------------------------------------------------------------
# 3. Compare models using LOO + BB-pseudo-BMA
# ------------------------------------------------------------
compare_df = az.compare(
    {
        "linear": trace_linear,
        "quadratic": trace_quad
    },
    method="BB-pseudo-BMA",
    ic="loo"
)

w_linear = compare_df.loc["linear", "weight"]
w_quad = compare_df.loc["quadratic", "weight"]

print(compare_df)
print(f"\nBMA weights:")
print(f"Linear:    {w_linear:.4f}")
print(f"Quadratic: {w_quad:.4f}")

# ------------------------------------------------------------
# 4. Posterior predictive samples
# ------------------------------------------------------------
with linear_model:
    ppc_linear = pm.sample_posterior_predictive(
        trace_linear,
        return_inferencedata=True,
        random_seed=42
    )

with quad_model:
    ppc_quad = pm.sample_posterior_predictive(
        trace_quad,
        return_inferencedata=True,
        random_seed=42
    )

y_samples_linear = (
    ppc_linear.posterior_predictive["y_obs"]
    .stack(draws=("chain", "draw"))
    .values
)

y_samples_quad = (
    ppc_quad.posterior_predictive["y_obs"]
    .stack(draws=("chain", "draw"))
    .values
)

# ------------------------------------------------------------
# 5. BMA posterior predictive mean
# ------------------------------------------------------------
y_pred_linear = y_samples_linear.mean(axis=1)
y_pred_quad = y_samples_quad.mean(axis=1)

y_pred_bma = (
    w_linear * y_pred_linear
    + w_quad * y_pred_quad
)

# ------------------------------------------------------------
# 6. Plot
# ------------------------------------------------------------
plt.figure(figsize=(9, 6))

plt.scatter(x, y, label="Observed")

plt.plot(
    x,
    y_pred_linear,
    label=f"Linear (w={w_linear:.2f})"
)

plt.plot(
    x,
    y_pred_quad,
    label=f"Quadratic (w={w_quad:.2f})"
)

plt.plot(
    x,
    y_pred_bma,
    linewidth=3,
    label="BMA prediction"
)

plt.xlabel("x")
plt.ylabel("y")
plt.title("Bayesian Model Averaging")
plt.legend()
plt.show()